In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler,OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report

In [2]:
df = pd.read_csv("loan_approval_dataset.csv")
df.head()

,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Approved
1,2,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Rejected
2,3,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Rejected
3,4,3,Graduate,No,8200000,30700000,8,467,18200000,3300000,23300000,7900000,Rejected
4,5,5,Not Graduate,Yes,9800000,24200000,20,382,12400000,8200000,29400000,5000000,Rejected


In [3]:
df.shape
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4269 entries, 0 to 4268
Data columns (total 13 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   loan_id                    4269 non-null   int64 
 1    no_of_dependents          4269 non-null   int64 
 2    education                 4269 non-null   object
 3    self_employed             4269 non-null   object
 4    income_annum              4269 non-null   int64 
 5    loan_amount               4269 non-null   int64 
 6    loan_term                 4269 non-null   int64 
 7    cibil_score               4269 non-null   int64 
 8    residential_assets_value  4269 non-null   int64 
 9    commercial_assets_value   4269 non-null   int64 
 10   luxury_assets_value       4269 non-null   int64 
 11   bank_asset_value          4269 non-null   int64 
 12   loan_status               4269 non-null   object
dtypes: int64(10), object(3)
memory usage: 433.7+ KB


,loan_id,no_of_dependents,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value
count,4269.000000,4269.000000,4.269000e+03,4.269000e+03,4269.000000,4269.000000,4.269000e+03,4.269000e+03,4.269000e+03,4.269000e+03
mean,2135.000000,2.498712,5.059124e+06,1.513345e+07,10.900445,599.936051,7.472617e+06,4.973155e+06,1.512631e+07,4.976692e+06
std,1232.498479,1.695910,2.806840e+06,9.043363e+06,5.709187,172.430401,6.503637e+06,4.388966e+06,9.103754e+06,3.250185e+06
min,1.000000,0.000000,2.000000e+05,3.000000e+05,2.000000,300.000000,-1.000000e+05,0.000000e+00,3.000000e+05,0.000000e+00
25%,1068.000000,1.000000,2.700000e+06,7.700000e+06,6.000000,453.000000,2.200000e+06,1.300000e+06,7.500000e+06,2.300000e+06
50%,2135.000000,3.000000,5.100000e+06,1.450000e+07,10.000000,600.000000,5.600000e+06,3.700000e+06,1.460000e+07,4.600000e+06
75%,3202.000000,4.000000,7.500000e+06,2.150000e+07,16.000000,748.000000,1.130000e+07,7.600000e+06,2.170000e+07,7.100000e+06
max,4269.000000,5.000000,9.900000e+06,3.950000e+07,20.000000,900.000000,2.910000e+07,1.940000e+07,3.920000e+07,1.470000e+07


In [4]:
df.isnull().sum()

loan_id                      0
 no_of_dependents            0
 education                   0
 self_employed               0
 income_annum                0
 loan_amount                 0
 loan_term                   0
 cibil_score                 0
 residential_assets_value    0
 commercial_assets_value     0
 luxury_assets_value         0
 bank_asset_value            0
 loan_status                 0
dtype: int64

In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
df = df.drop("loan_id",axis=1)

In [7]:
df.columns = df.columns.str.strip()

In [8]:
asscols = ["residential_assets_value","commercial_assets_value","luxury_assets_value","bank_asset_value"]
df["assets"] = df[asscols].sum(axis=1)
df.drop(columns=asscols, inplace=True)
df.head()

,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,loan_status,assets
0,2,Graduate,No,9600000,29900000,12,778,Approved,50700000
1,0,Not Graduate,Yes,4100000,12200000,8,417,Rejected,17000000
2,3,Graduate,No,9100000,29700000,20,506,Rejected,57700000
3,3,Graduate,No,8200000,30700000,8,467,Rejected,52700000
4,5,Not Graduate,Yes,9800000,24200000,20,382,Rejected,55000000


In [9]:
cat_cols = df.select_dtypes(include="object").columns
for col in cat_cols:
    print(df[col].value_counts())
    print("-----------")

education
Graduate        2144
Not Graduate    2125
Name: count, dtype: int64
-----------
self_employed
Yes    2150
No     2119
Name: count, dtype: int64
-----------
loan_status
Approved    2656
Rejected    1613
Name: count, dtype: int64
-----------


In [10]:
for col in cat_cols:
    df[col] = df[col].str.strip()
    print(df[col].unique())
    print("-----------")

['Graduate' 'Not Graduate']
-----------
['No' 'Yes']
-----------
['Approved' 'Rejected']
-----------


In [11]:
df['education'] = df['education'].replace({
    'Graduate': 1,
    'Not Graduate': 0
})

df['self_employed'] = df['self_employed'].replace({
    'Yes': 1,
    'No': 0
})

df['loan_status'] = df['loan_status'].replace({
    'Approved': 1,
    'Rejected': 0
})

df.head()

C:\Users\DELL\AppData\Local\Temp\ipykernel_25712\3814508119.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['education'] = df['education'].replace({
C:\Users\DELL\AppData\Local\Temp\ipykernel_25712\3814508119.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['self_employed'] = df['self_employed'].replace({
C:\Users\DELL\AppData\Local\Temp\ipykernel_25712\3814508119.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `

,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,loan_status,assets
0,2,1,0,9600000,29900000,12,778,1,50700000
1,0,0,1,4100000,12200000,8,417,0,17000000
2,3,1,0,9100000,29700000,20,506,0,57700000
3,3,1,0,8200000,30700000,8,467,0,52700000
4,5,0,1,9800000,24200000,20,382,0,55000000


In [12]:
X = df.drop(columns=["loan_status"])
y = df["loan_status"]

X_remaining, X_unseen, y_remaining, y_unseen = train_test_split(
    X,
    y,
    test_size=0.1,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X_remaining,
    y_remaining,
    test_size=0.2,
    random_state=42
)

num_cols = X.select_dtypes(include=["int","float"]).columns.tolist()

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_unseen = scaler.transform(X_unseen)

In [13]:
lr = LogisticRegression()
lr.fit(X_train,y_train)
lrp = lr.predict(X_test)
lrpu = lr.predict(X_unseen)
print("Train : ",accuracy_score(y_test,lrp))
print("Unseen : ",accuracy_score(y_unseen,lrpu))
print(classification_report(y_unseen, lrpu))
print(confusion_matrix(y_unseen,lrpu))

Train :  0.9206762028608583
Unseen :  0.9110070257611241
              precision    recall  f1-score   support

           0       0.88      0.88      0.88       163
           1       0.93      0.93      0.93       264

    accuracy                           0.91       427
   macro avg       0.91      0.91      0.91       427
weighted avg       0.91      0.91      0.91       427

[[144  19]
 [ 19 245]]


In [14]:
dt = DecisionTreeClassifier(random_state=56)
dt.fit(X_train,y_train)
dtp = dt.predict(X_test)
dtpu = dt.predict(X_unseen)
print("Train : ",accuracy_score(y_test,dtp))
print("Unseen : ",accuracy_score(y_unseen,dtpu))
print(classification_report(y_unseen, dtpu))
print(confusion_matrix(y_unseen,dtpu))

Train :  0.9869960988296489
Unseen :  0.9765807962529274
              precision    recall  f1-score   support

           0       0.98      0.96      0.97       163
           1       0.98      0.98      0.98       264

    accuracy                           0.98       427
   macro avg       0.98      0.97      0.98       427
weighted avg       0.98      0.98      0.98       427

[[157   6]
 [  4 260]]


In [15]:
rf = RandomForestClassifier(n_estimators=100,max_depth=5,min_samples_split=3,random_state=23)
rf.fit(X_train,y_train)
rfp = rf.predict(X_test)
rfpu = rf.predict(X_unseen)
print("Train : ",accuracy_score(y_test,rfp))
print("Unseen : ",accuracy_score(y_unseen,rfpu))
print(classification_report(y_unseen, rfpu))
print(confusion_matrix(y_unseen,rfpu))

Train :  0.9752925877763329
Unseen :  0.9578454332552693
              precision    recall  f1-score   support

           0       0.96      0.93      0.94       163
           1       0.96      0.97      0.97       264

    accuracy                           0.96       427
   macro avg       0.96      0.95      0.96       427
weighted avg       0.96      0.96      0.96       427

[[152  11]
 [  7 257]]


In [17]:
import pickle
with open("loan_model.pkl","wb") as file:
    pickle.dump(dt,file)
with open("scaler.pkl","wb") as file:
    pickle.dump(scaler,file)